In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [ ]:
# Best F1 per benchmark
for bench in df.index.get_level_values("benchmark").unique():
    group = df.loc[bench]
    best = group.loc[group["f1_score"].idxmax()]
    print(
        f"{bench}: F1={best['f1_score']:.4f} | k={best['k']}, widths={best['widths']}, aux_loss={best['aux_loss']}, lr={best['topk_threshold_lr']}"
    )

In [ ]:
DISTRIBUTION = "HIERARCHICAL_PAIRS"

In [ ]:
df = pd.read_parquet("data/matryoshka/results.parquet")
cp = df.loc[DISTRIBUTION].reset_index()
cp["seed"] = cp["seed"].astype(int)
cp["k"] = cp["k"].astype(int)
cp["aux_loss"] = cp["aux_loss"].map({True: "Yes", False: "No"})
cp["topk_threshold_lr"] = cp["topk_threshold_lr"].astype(str)
cp.shape

## Marginal effects on F1 score

In [ ]:
# Marginal box plots: F1 vs each hyperparameter
fig = make_subplots(
    rows=1,
    cols=4,
    subplot_titles=["k", "Nesting Depth", "Auxiliary Loss", "TopK Threshold LR"],
)

for i, (col, labels) in enumerate(
    [
        ("k", None),
        ("widths", None),
        ("aux_loss", None),
        ("topk_threshold_lr", None),
    ]
):
    for seed in sorted(cp["seed"].unique()):
        sub = cp[cp["seed"] == seed]
        fig.add_trace(
            go.Box(
                x=sub[col].astype(str),
                y=sub["f1_score"],
                name=f"seed {seed}",
                legendgroup=f"seed {seed}",
                showlegend=(i == 0),
                marker_color=px.colors.qualitative.Plotly[seed],
                boxpoints="all",
                jitter=0.3,
                pointpos=0,
            ),
            row=1,
            col=i + 1,
        )

fig.update_layout(
    height=450,
    width=1200,
    title_text="F1 Score by Hyperparameter (Correlated Pairs, all seeds)",
    boxmode="group",
)
fig.show()

## Seed consistency check

In [ ]:
# Seed variance: how much does F1 vary across seeds for each SAE config?
seed_var = (
    cp.groupby("sae")["f1_score"]
    .agg(["mean", "std", "count"])
    .sort_values("std", ascending=False)
)
print(
    f"Max seed std: {seed_var['std'].max():.4f}, Mean seed std: {seed_var['std'].mean():.4f}"
)

fig = px.histogram(
    seed_var,
    x="std",
    nbins=30,
    title="Distribution of F1 std across seeds (per SAE config)",
)
fig.update_layout(
    xaxis_title="F1 std across 3 seeds", yaxis_title="Count of SAE configs"
)
fig.show()

## Interaction: F1 vs k, faceted by nesting depth and auxiliary loss

In [ ]:
# F1 vs k, faceted by nesting depth (rows) and aux_loss (columns), colored by LR
fig = px.strip(
    cp,
    x="k",
    y="f1_score",
    color="topk_threshold_lr",
    facet_row="widths",
    facet_col="aux_loss",
    category_orders={
        "widths": ["2-level", "3-level", "4-level"],
        "aux_loss": ["No", "Yes"],
    },
    labels={"f1_score": "F1 Score", "k": "k", "topk_threshold_lr": "LR"},
    title="F1 vs k | rows: nesting depth, cols: aux loss, color: LR",
    height=600,
    width=900,
)
fig.update_traces(marker_size=5, jitter=0.4)
fig.show()

## Interaction heatmaps (mean F1)

In [ ]:
# Pairwise interaction heatmaps for mean F1
pairs = [
    ("k", "widths"),
    ("k", "aux_loss"),
    ("k", "topk_threshold_lr"),
    ("widths", "topk_threshold_lr"),
]
fig = make_subplots(
    rows=1,
    cols=4,
    subplot_titles=[f"{a} x {b}" for a, b in pairs],
)

for i, (col_a, col_b) in enumerate(pairs):
    pivot = cp.pivot_table(
        values="f1_score", index=col_b, columns=col_a, aggfunc="mean"
    )
    pivot = pivot.sort_index(ascending=False)
    fig.add_trace(
        go.Heatmap(
            z=pivot.values,
            x=[str(c) for c in pivot.columns],
            y=[str(r) for r in pivot.index],
            colorscale="Viridis",
            showscale=(i == len(pairs) - 1),
            text=pivot.values.round(3),
            texttemplate="%{text}",
        ),
        row=1,
        col=i + 1,
    )

fig.update_layout(
    height=350,
    width=1200,
    title_text="Mean F1 Score — Pairwise Interactions (Correlated Pairs)",
)
fig.show()

## F1 vs k line plot (mean over seeds, colored by LR, faceted by depth x aux_loss)

In [ ]:
# Line plot: mean F1 vs k, with error bars from seeds
agg = (
    cp.groupby(["k", "widths", "aux_loss", "topk_threshold_lr"])["f1_score"]
    .agg(["mean", "std"])
    .reset_index()
)
agg.columns = ["k", "widths", "aux_loss", "topk_threshold_lr", "f1_mean", "f1_std"]

fig = px.line(
    agg,
    x="k",
    y="f1_mean",
    error_y="f1_std",
    color="topk_threshold_lr",
    facet_row="widths",
    facet_col="aux_loss",
    category_orders={
        "widths": ["2-level", "3-level", "4-level"],
        "aux_loss": ["No", "Yes"],
    },
    markers=True,
    labels={"f1_mean": "Mean F1", "k": "k", "topk_threshold_lr": "LR"},
    title="Mean F1 (± std across seeds) vs k",
    height=650,
    width=900,
)
fig.show()

## Best HIERARCHICAL_PAIRS result — training loss

In [ ]:
# Find best HIERARCHICAL_PAIRS config
hp = df.loc["HIERARCHICAL_PAIRS"]
best_hp = hp.loc[hp["f1_score"].idxmax()]
best_sae = best_hp.name[0]  # sae name from (sae, seed) index
print(f"Best SAE: {best_sae}")
print(
    f"F1={best_hp['f1_score']:.4f} | k={best_hp['k']}, widths={best_hp['widths']}, aux_loss={best_hp['aux_loss']}, lr={best_hp['topk_threshold_lr']}"
)

# Load losses and plot for this SAE on HIERARCHICAL_PAIRS
losses = pd.read_parquet("data/matryoshka/losses.parquet")
best_losses = losses[
    (losses["benchmark"] == "HIERARCHICAL_PAIRS") & (losses["sae"] == best_sae)
]

fig = px.line(
    best_losses,
    x="step",
    y="loss",
    color="seed",
    title=f"Training Loss — {best_sae} (HIERARCHICAL_PAIRS, F1={best_hp['f1_score']:.4f})",
    labels={"step": "Step", "loss": "Loss", "seed": "Seed"},
    height=400,
    width=800,
)
fig.show()

In [ ]:
df = pd.read_parquet("data/matryoshka/results.parquet")
df.loc["SPARSE_UNIFORM"]
# df.loc[df["f1_score"].idxmax()]

In [ ]:
df.loc["CORRELATED_PAIRS"]  # all SAEs, all metrics
df.loc["CORRELATED_PAIRS"].xs("0", level="seed")
df.loc["CORRELATED_PAIRS", "Standard"]["f1_score"]
df.loc["CORRELATED_PAIRS"].loc[["Standard", "Matryoshka"]]
df[["f1_score", "mcc"]]

In [ ]:
df = pd.read_parquet("data/matryoshka/results.parquet")

In [ ]:
df.loc["HIERARCHICAL_PAIRS"]